# Distributed IoT Intrusion Detection and Network Traffic Analytics
**Dataset:** RT-IoT2022 (`RT_IOT2022.csv`), 123,117 flows, 85 attributes. Real file only; no synthetic resampling.
**Platform:** PySpark 3.5 standalone cluster (one master, one worker offering 4 cores) in Google Colab or Linux.

| Team member | Module | PySpark API |
|---|---|---|
| Beven Mpofu | Clean, engineer, encode, scale | `StringIndexer`, `OneHotEncoder`, `VectorAssembler`, `StandardScaler` |
| Memory Mhembere | Group summaries and Pearson correlation | `pyspark.sql.functions`, `Correlation.corr` |
| Sibonginkosi T Nkashe | Multiclass logistic regression, 80/20 split | `LogisticRegression`, `MulticlassClassificationEvaluator` |

Every module is timed at 1, 2, and 4 executors (1 core each), 3 repeats, with the partition count fixed. Speedup is \(T_1/T_n\). In Google Colab, upload `RT_IOT2022.csv` to `/content/`. From a clone of this repository, the notebook reads `data/RT_IOT2022.csv` and writes tables to `outputs/` and figures to `outputs/figures/`.


## 1 — Install Spark and start the standalone cluster
The worker advertises 4 cores so a session can take 1, 2, or 4 executors. Driver memory is fixed when the JVM starts, so it is set here. If this machine has fewer than 4 physical cores, 4 executors oversubscribe the CPU; record `os.cpu_count()` in the report.


In [ ]:
import os, sys, site, socket, subprocess, time, shutil

user_site = site.getusersitepackages()
if user_site not in sys.path:
    sys.path.append(user_site)

java_home = os.environ.get("JAVA_HOME", "")
java_candidates = [java_home, os.path.expanduser("~/opt/jdk-17"), "/usr/lib/jvm/java-17-openjdk-amd64"]
for candidate in java_candidates:
    java_bin = os.path.join(candidate, "bin", "java") if candidate else ""
    if java_bin and os.path.isfile(java_bin):
        os.environ["JAVA_HOME"] = candidate
        os.environ["PATH"] = os.path.join(candidate, "bin") + os.pathsep + os.environ.get("PATH", "")
        break
if shutil.which("java") is None:
    raise RuntimeError("Java is required. In Colab it is preinstalled; locally set JAVA_HOME.")

pip_base = [sys.executable, "-m", "pip", "-q", "install", "pyspark==3.5.1", "psutil", "matplotlib", "pandas", "numpy"]
installed = False
for extra in ([], ["--user"], ["--user", "--break-system-packages"], ["--break-system-packages"]):
    if subprocess.run(pip_base + extra).returncode == 0:
        installed = True
        break
if not installed:
    raise RuntimeError("pip could not install pyspark==3.5.1. On Colab the plain pip install is enough.")

import pyspark
SPARK_HOME = os.path.dirname(pyspark.__file__)
MASTER_URL = "spark://127.0.0.1:7077"
os.environ["SPARK_HOME"] = SPARK_HOME
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_SUBMIT_ARGS"] = "--driver-memory 1g pyspark-shell"

master_script = os.path.join(SPARK_HOME, "sbin", "start-master.sh")
worker_script = os.path.join(SPARK_HOME, "sbin", "start-worker.sh")
# The current PyPI sdist of pyspark 3.5.1 ships jars but not the standalone
# start scripts. Pull the matching 3.5.1 scripts if they are absent.
if not os.path.exists(master_script) or not os.path.exists(worker_script):
    import urllib.request
    base = "https://raw.githubusercontent.com/apache/spark/v3.5.1/sbin/"
    for name in ("start-master.sh", "start-worker.sh", "stop-master.sh", "stop-worker.sh"):
        dest = os.path.join(SPARK_HOME, "sbin", name)
        if not os.path.exists(dest):
            urllib.request.urlretrieve(base + name, dest)
        os.chmod(dest, 0o755)
if not os.path.exists(master_script):
    raise RuntimeError(
        "Spark's cluster scripts are Linux shell scripts. Run this notebook in Google Colab or WSL."
    )

def port_open(port):
    s = socket.socket()
    s.settimeout(1)
    try:
        s.connect(("127.0.0.1", port))
        return True
    except OSError:
        return False
    finally:
        s.close()

if not port_open(7077):
    subprocess.run(f"{master_script} -h 127.0.0.1", shell=True, check=False)
    time.sleep(5)
    worker_script = os.path.join(SPARK_HOME, "sbin", "start-worker.sh")
    subprocess.run(
        f"SPARK_WORKER_CORES=4 SPARK_WORKER_MEMORY=4g {worker_script} {MASTER_URL}",
        shell=True,
        check=False,
    )
    time.sleep(5)
print("Cluster ready. Master:", MASTER_URL)
print("Java:", os.environ.get("JAVA_HOME", "from PATH"))


## 2 — Configuration
Partition count stays at 8 for every trial, and adaptive query execution is off, so the only change between trials is the number of executors. `IOT_EXECUTORS` and `IOT_REPEATS` override the proposal defaults (1,2,4 and 3) for a shorter rerun.


In [ ]:
import json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psutil

from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.stat import Correlation
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

NUM_PARTITIONS = 8
EXECUTOR_LIST = [int(x) for x in os.environ.get("IOT_EXECUTORS", "1,2,4").split(",") if x]
REPEATS = int(os.environ.get("IOT_REPEATS", "3"))
SEED = 42
MAX_ITER = 50

CAT_COLS = ["proto", "service"]
LABEL_COL = "Attack_type"
DROP_COLS = ["no", "bwd_URG_flag_count"]
PROFILE_COLS = [
    "flow_duration", "flow_pkts_per_sec", "bytes_per_pkt", "flag_events",
    "total_pkts", "total_payload", "down_up_ratio", "flow_SYN_flag_count",
]
# Normal device profiles in RT-IoT2022. Every other Attack_type value is an attack.
# The column keeps that name for benign rows too; the model predicts the raw label.
BENIGN = {"Thing_Speak", "MQTT_Publish", "Wipro_bulb"}

def find_repo_root():
    start = os.path.abspath(os.getcwd())
    for base in [start, os.path.dirname(start), os.path.dirname(os.path.dirname(start))]:
        if os.path.isfile(os.path.join(base, "data", "RT_IOT2022.csv")):
            return base
    return start

REPO_ROOT = find_repo_root()
CANDIDATES = [
    "/content/RT_IOT2022.csv",
    os.path.join(REPO_ROOT, "data", "RT_IOT2022.csv"),
    os.path.join(os.getcwd(), "data", "RT_IOT2022.csv"),
    os.path.join(os.getcwd(), "RT_IOT2022.csv"),
    os.path.expanduser("~/data/RT_IOT2022.csv"),
]
DATA_PATH = next((p for p in CANDIDATES if os.path.exists(p)), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Place RT_IOT2022.csv in /content/ on Colab, or in data/ at the repository root."
    )

if os.path.isdir("/content") and not os.path.isdir(os.path.join(REPO_ROOT, "outputs")):
    OUT_DIR = "/content"
else:
    OUT_DIR = os.path.join(REPO_ROOT, "outputs")
os.makedirs(OUT_DIR, exist_ok=True)
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

def show(obj):
    try:
        display(obj)
    except NameError:
        print(obj.to_string() if hasattr(obj, "to_string") else obj)

print("Data:", DATA_PATH)
print("Outputs:", OUT_DIR)
print("Executors:", EXECUTOR_LIST, "repeats:", REPEATS)


## 3 — Load, clean, engineer, encode, scale
Decisions taken from the file itself:

- No nulls. `dropna` stays as a guard and should remove nothing.
- `no` is a row index. Drop it. Do not drop duplicate flows after that: 5,195 records are identical once the index is gone, and they are repeated measurements (mostly the SYN flood), not bad rows. The proposal uses all 123,117 records.
- `bwd_URG_flag_count` is constant 0. A standard deviation of 0 would wipe that column inside `StandardScaler`. Drop it.
- Column names such as `fwd_pkts_payload.min` contain `.`, which Spark treats as a nested field. Rename `.` to `_` with `toDF` before any cast.
- `service == "-"` is Zeek's sentinel for an unidentified service (102,861 rows). Rewrite it to `unknown`.
- Do not clip extreme rates or durations. `DOS_SYN_Hping` is 94,659 rows; a packet rate near 2e6 is the attack, not a sensor error.
- Encode `proto` (3 levels) and `service` (10 levels) only. `id_orig_p` has 65,478 values and `id_resp_p` has 1,809, so both stay numeric. One-hot encoding either port would explode the feature width.
- `Attack_type` is the label (12 classes, including the file's spelling `ARP_poisioning`). It is not a feature. Pearson correlation against the integer label is not used, because that index is nominal.

Engineered columns: `total_pkts`, `total_payload`, `bytes_per_pkt`, `flag_events`, `log1p_flow_duration`, `log1p_flow_pkts_per_sec`.
The model matrix is a dense scaled numeric vector concatenated with the two one-hot vectors.


In [ ]:
INT_COLS = {
    "no", "id_orig_p", "id_resp_p",
    "fwd_pkts_tot", "bwd_pkts_tot", "fwd_data_pkts_tot", "bwd_data_pkts_tot",
    "fwd_header_size_tot", "fwd_header_size_min", "fwd_header_size_max",
    "bwd_header_size_tot", "bwd_header_size_min", "bwd_header_size_max",
    "flow_FIN_flag_count", "flow_SYN_flag_count", "flow_RST_flag_count",
    "fwd_PSH_flag_count", "bwd_PSH_flag_count", "flow_ACK_flag_count",
    "fwd_URG_flag_count", "bwd_URG_flag_count", "flow_CWR_flag_count", "flow_ECE_flag_count",
    "fwd_init_window_size", "bwd_init_window_size", "fwd_last_window_size",
}
STR_COLS = {"proto", "service", "Attack_type"}
NEEDED = [
    "proto", "service", "Attack_type", "flow_duration", "fwd_pkts_tot", "bwd_pkts_tot",
    "fwd_pkts_payload_tot", "bwd_pkts_payload_tot", "flow_pkts_per_sec",
    "flow_FIN_flag_count", "flow_SYN_flag_count", "flow_RST_flag_count",
    "fwd_PSH_flag_count", "bwd_PSH_flag_count", "flow_ACK_flag_count",
    "down_up_ratio", "payload_bytes_per_second",
]

def load_raw(spark):
    df = spark.read.csv(DATA_PATH, header=True, inferSchema=False, mode="FAILFAST")
    df = df.toDF(*[c.replace(".", "_") for c in df.columns])
    if len(df.columns) != 85:
        raise ValueError(f"Expected 85 columns, found {len(df.columns)}")
    missing = [c for c in NEEDED if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns after rename: {missing}")
    for c in df.columns:
        if c in STR_COLS:
            continue
        df = df.withColumn(c, F.col(c).cast("int" if c in INT_COLS else "double"))
    return df.repartition(NUM_PARTITIONS)

def clean(df):
    present = [c for c in DROP_COLS if c in df.columns]
    df = df.drop(*present)
    df = df.withColumn(
        "service",
        F.when(F.col("service") == "-", F.lit("unknown")).otherwise(F.col("service")),
    )
    return df.dropna()

def engineer(df):
    df = df.withColumn("total_pkts", F.col("fwd_pkts_tot") + F.col("bwd_pkts_tot"))
    df = df.withColumn(
        "total_payload",
        F.col("fwd_pkts_payload_tot") + F.col("bwd_pkts_payload_tot"),
    )
    df = df.withColumn(
        "bytes_per_pkt",
        F.when(F.col("total_pkts") > 0, F.col("total_payload") / F.col("total_pkts")).otherwise(F.lit(0.0)),
    )
    df = df.withColumn(
        "flag_events",
        F.col("flow_FIN_flag_count") + F.col("flow_SYN_flag_count") + F.col("flow_RST_flag_count")
        + F.col("fwd_PSH_flag_count") + F.col("bwd_PSH_flag_count") + F.col("flow_ACK_flag_count"),
    )
    df = df.withColumn("log1p_flow_duration", F.log1p(F.col("flow_duration")))
    df = df.withColumn("log1p_flow_pkts_per_sec", F.log1p(F.col("flow_pkts_per_sec")))
    return df

def numeric_columns(df):
    skip = set(CAT_COLS) | {LABEL_COL, "label", "weight"}
    numeric_types = {"int", "bigint", "double", "float", "long", "smallint", "tinyint"}
    return [c for c, t in df.dtypes if t in numeric_types and c not in skip]

def build_pipeline(num_cols):
    """Index the label, one-hot encode proto and service, scale the numeric block, assemble features."""
    label_idx = StringIndexer(
        inputCol=LABEL_COL, outputCol="label", handleInvalid="keep", stringOrderType="frequencyDesc"
    )
    cat_idx = StringIndexer(
        inputCols=CAT_COLS, outputCols=["proto_idx", "service_idx"], handleInvalid="keep"
    )
    # dropLast=False keeps every level. regParam on the classifier absorbs the extra intercept column.
    # handleInvalid="keep" covers a service level that randomSplit leaves out of the training fold
    # (service "radius" has only 2 rows).
    ohe = OneHotEncoder(
        inputCols=["proto_idx", "service_idx"],
        outputCols=["proto_ohe", "service_ohe"],
        dropLast=False,
        handleInvalid="keep",
    )
    num_asm = VectorAssembler(inputCols=num_cols, outputCol="numeric_raw")
    scaler = StandardScaler(
        inputCol="numeric_raw", outputCol="numeric_scaled", withMean=True, withStd=True
    )
    features = VectorAssembler(
        inputCols=["numeric_scaled", "proto_ohe", "service_ohe"], outputCol="features"
    )
    return Pipeline(stages=[label_idx, cat_idx, ohe, num_asm, scaler, features])

def run_preprocessing(spark):
    """Beven: clean, engineer, encode, scale. The whole function is the timed region."""
    raw = load_raw(spark)
    n_raw = raw.count()
    parts = raw.rdd.getNumPartitions()
    df = engineer(clean(raw)).cache()
    n_clean = df.count()
    num_cols = numeric_columns(df)
    model = build_pipeline(num_cols).fit(df)
    out = model.transform(df)
    balance = df.groupBy(LABEL_COL).count().orderBy(F.desc("count")).collect()
    out.select("features", "label").write.format("noop").mode("overwrite").save()
    dim = int(out.select("features").first()["features"].size)
    df.unpersist()
    return {
        "rows_raw": int(n_raw),
        "rows_clean": int(n_clean),
        "rows_removed": int(n_raw - n_clean),
        "n_numeric": len(num_cols),
        "feature_dim": dim,
        "n_classes": len(model.stages[0].labels),
        "label_order": list(model.stages[0].labels),
        "class_counts": [(r[LABEL_COL], int(r["count"])) for r in balance],
        "partitions": int(parts),
    }


## 4 — Group statistics and Pearson correlation
Summaries are grouped by `Attack_type` (and by `proto` × `service`). The Pearson matrix is computed on every numeric column whose standard deviation is positive. Features are then ranked by how far apart the class means are, relative to the column's overall standard deviation. That rank, together with the strongest pairs, is the anomaly-indicator list. Correlation against the class index is omitted because the index is an arbitrary code.


In [ ]:
def run_statistics(spark):
    """Memory: group summaries + Correlation.corr(method='pearson')."""
    raw = load_raw(spark)
    parts = raw.rdd.getNumPartitions()
    df = engineer(clean(raw)).cache()
    n_clean = df.count()
    num_cols = numeric_columns(df)

    std_row = df.agg(*[F.stddev(c).alias(c) for c in num_cols]).first()
    corr_cols = [c for c in num_cols if std_row[c] is not None and float(std_row[c]) > 1e-12]

    profile_aggs = [F.count("*").alias("n")]
    for c in PROFILE_COLS:
        profile_aggs.append(F.mean(c).alias(f"avg_{c}"))
        profile_aggs.append(F.stddev(c).alias(f"sd_{c}"))
    profile = [
        row.asDict()
        for row in df.groupBy(LABEL_COL).agg(*profile_aggs).orderBy(F.desc("n")).collect()
    ]
    mix = [
        row.asDict()
        for row in df.groupBy("proto", "service").count().orderBy(F.desc("count")).collect()
    ]

    mean_rows = df.groupBy(LABEL_COL).agg(*[F.mean(c).alias(c) for c in corr_cols]).collect()
    separators = []
    for c in corr_cols:
        vals = [float(r[c]) for r in mean_rows if r[c] is not None]
        if len(vals) < 2:
            continue
        spread = float(np.std(vals))
        gstd = float(std_row[c])
        separators.append((c, spread / gstd if gstd else 0.0))
    separators.sort(key=lambda t: t[1], reverse=True)

    assembled = VectorAssembler(inputCols=corr_cols, outputCol="corr_features").transform(df)
    matrix = Correlation.corr(assembled, "corr_features", "pearson").head()[0].toArray()
    pairs = []
    n = len(corr_cols)
    for i in range(n):
        for j in range(i + 1, n):
            r = float(matrix[i, j])
            if math.isnan(r):
                continue
            pairs.append((abs(r), r, corr_cols[i], corr_cols[j]))
    pairs.sort(reverse=True)
    top_pairs = [
        {"feature_a": a, "feature_b": b, "pearson_r": r}
        for _, r, a, b in pairs[:20]
    ]
    df.unpersist()
    return {
        "rows_clean": int(n_clean),
        "n_features_corr": len(corr_cols),
        "partitions": int(parts),
        "profile": profile,
        "proto_service": mix,
        "separators": [{"feature": c, "class_mean_spread": s} for c, s in separators[:20]],
        "top_pairs": top_pairs,
        "corr_cols": corr_cols,
        "corr": matrix.tolist(),
    }


## 5 — Multiclass logistic regression
The feature pipeline is fit on the training fold only. `family="multinomial"` is required for 12 `Attack_type` classes. `standardization` is off because `StandardScaler` already produced the numeric block. `MulticlassClassificationEvaluator` reports accuracy and weighted F1. Per-class precision and recall are computed from the confusion matrix so a 77% `DOS_SYN_Hping` share cannot hide a model that never detects the rare scans. The baseline is "always predict the majority training class."


In [ ]:
def run_classification(spark):
    """Sibonginkosi: multinomial logistic regression, 80/20 split."""
    raw = load_raw(spark)
    parts = raw.rdd.getNumPartitions()
    df = engineer(clean(raw)).cache()
    n_clean = df.count()
    num_cols = numeric_columns(df)
    train, test = df.randomSplit([0.8, 0.2], seed=SEED)

    model = build_pipeline(num_cols).fit(train)
    train_t = model.transform(train).select("features", "label").cache()
    test_t = model.transform(test).select("features", "label").cache()
    n_train = train_t.count()
    n_test = test_t.count()
    dim = int(train_t.first()["features"].size)

    lr = LogisticRegression(
        featuresCol="features",
        labelCol="label",
        family="multinomial",
        maxIter=MAX_ITER,
        regParam=0.01,
        elasticNetParam=0.0,
        standardization=False,
        tol=1e-6,
    )
    pred = lr.fit(train_t).transform(test_t).cache()
    pred.count()

    def evaluate(metric):
        return float(MulticlassClassificationEvaluator(
            labelCol="label", predictionCol="prediction", metricName=metric
        ).evaluate(pred))

    accuracy = evaluate("accuracy")
    f1 = evaluate("f1")
    wprec = evaluate("weightedPrecision")
    wrec = evaluate("weightedRecall")

    labels = list(model.stages[0].labels)
    k = len(labels)
    cm = [[0] * k for _ in range(k)]
    for row in pred.groupBy("label", "prediction").count().collect():
        i, j = int(row["label"]), int(row["prediction"])
        if 0 <= i < k and 0 <= j < k:
            cm[i][j] += int(row["count"])

    per_class = []
    f1s = []
    for i, name in enumerate(labels):
        tp = cm[i][i]
        support = sum(cm[i])
        predicted = sum(cm[r][i] for r in range(k))
        prec = tp / predicted if predicted else 0.0
        rec = tp / support if support else 0.0
        fscore = (2 * prec * rec / (prec + rec)) if (prec + rec) else 0.0
        f1s.append(fscore)
        per_class.append({
            "class": name,
            "precision": prec,
            "recall": rec,
            "f1": fscore,
            "support": int(support),
            "role": "benign" if name in BENIGN else "attack",
        })

    train_counts = train.groupBy(LABEL_COL).count().collect()
    majority_name = max(train_counts, key=lambda r: r["count"])[LABEL_COL]
    maj_idx = labels.index(majority_name) if majority_name in labels else 0
    covered = sum(sum(row) for row in cm)
    baseline = (sum(cm[maj_idx]) / covered) if covered else 0.0

    for frame in (df, train_t, test_t, pred):
        frame.unpersist()
    return {
        "rows_clean": int(n_clean),
        "train_rows": int(n_train),
        "test_rows": int(n_test),
        "feature_dim": dim,
        "partitions": int(parts),
        "accuracy": accuracy,
        "weighted_f1": f1,
        "weighted_precision": wprec,
        "weighted_recall": wrec,
        "macro_f1": float(sum(f1s) / len(f1s)) if f1s else 0.0,
        "baseline_accuracy": float(baseline),
        "majority_class": majority_name,
        "labels": labels,
        "per_class": per_class,
        "confusion": cm,
        "test_rows_in_matrix": int(covered),
    }


## 6 — Timing harness
Cluster startup and the Python-worker warm-up are outside the clock. The clock covers one module's full script, including the CSV read, which is part of that script.


In [ ]:
MEMBERS = {
    "Beven: Preprocessing": run_preprocessing,
    "Memory: Correlation": run_statistics,
    "Sibonginkosi: Logistic Reg.": run_classification,
}

def make_spark(n_exec):
    spark = (
        SparkSession.builder
        .master(MASTER_URL)
        .appName(f"iot_{n_exec}exec")
        .config("spark.executor.cores", "1")
        .config("spark.cores.max", str(n_exec))
        .config("spark.executor.memory", "512m")
        .config("spark.executor.memoryOverhead", "256m")
        .config("spark.sql.shuffle.partitions", str(NUM_PARTITIONS))
        .config("spark.default.parallelism", str(NUM_PARTITIONS))
        .config("spark.sql.adaptive.enabled", "false")
        .config("spark.driver.host", "127.0.0.1")
        .config("spark.driver.bindAddress", "127.0.0.1")
        .config("spark.ui.showConsoleProgress", "false")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("ERROR")
    return spark

def count_executors(spark):
    infos = spark.sparkContext._jsc.sc().statusTracker().getExecutorInfos()
    return len(infos) - 1

def warm_up(spark, n_exec, timeout=120):
    spark.sparkContext.parallelize(range(n_exec * 4), n_exec * 4).map(lambda x: x).count()
    t0 = time.time()
    while count_executors(spark) < n_exec and time.time() - t0 < timeout:
        time.sleep(1)

def stop_spark(spark):
    try:
        spark.stop()
    finally:
        SparkSession._instantiatedSession = None
        active = getattr(SparkSession, "_activeSession", None)
        if active is not None and not hasattr(active, "get"):
            SparkSession._activeSession = None

def timed_run(fn, n_exec):
    spark = make_spark(n_exec)
    try:
        warm_up(spark, n_exec)
        actual = count_executors(spark)
        t0 = time.perf_counter()
        metrics = fn(spark)
        elapsed = time.perf_counter() - t0
    finally:
        stop_spark(spark)
    return elapsed, actual, metrics

hw = {
    "os.cpu_count": os.cpu_count(),
    "ram_gb": round(psutil.virtual_memory().total / 1e9, 2),
    "partitions": NUM_PARTITIONS,
    "replication": 1,
    "executor_cores": 1,
    "repeats": REPEATS,
    "max_iter": MAX_ITER,
}
print("Hardware:", hw)
with open(os.path.join(OUT_DIR, "iot_hardware.json"), "w", encoding="utf-8") as f:
    json.dump(hw, f, indent=2)


## 7 — Executor scaling experiment
3 modules × 3 executor counts × 3 repeats = 27 timed jobs. The submitted 2-CPU run took about 45 minutes.


In [ ]:
records = []
last_metrics = {}
for member, fn in MEMBERS.items():
    for n_exec in EXECUTOR_LIST:
        for rep in range(1, REPEATS + 1):
            secs, actual, metrics = timed_run(fn, n_exec)
            records.append({
                "member": member,
                "executors": n_exec,
                "repeat": rep,
                "seconds": secs,
                "executors_seen": actual,
                "partitions": metrics.get("partitions"),
            })
            last_metrics[member] = metrics
            flag = "" if actual >= n_exec else "  WARNING: fewer executors than requested"
            print(
                f"{member:28s} executors={n_exec} (seen {actual}) rep={rep} -> {secs:7.2f}s{flag}",
                flush=True,
            )

raw = pd.DataFrame(records)
raw_path = os.path.join(OUT_DIR, "iot_raw_timings.csv")
raw.to_csv(raw_path, index=False)
print("Wrote", raw_path)


## 8 — Mean ± SD, speedup, efficiency
Efficiency is speedup divided by the executor count. This is the team table.


In [ ]:
summary = raw.groupby(["member", "executors"])["seconds"].agg(["mean", "std"]).reset_index()
t1 = summary[summary.executors == 1].set_index("member")["mean"]
summary["speedup"] = summary.apply(lambda r: t1[r["member"]] / r["mean"], axis=1)
summary["efficiency"] = summary["speedup"] / summary["executors"]
summary["mean ± SD (s)"] = summary.apply(lambda r: f"{r['mean']:.2f} ± {r['std']:.2f}", axis=1)
summary_path = os.path.join(OUT_DIR, "iot_summary.csv")
summary.to_csv(summary_path, index=False)

team = summary.pivot(index="member", columns="executors", values="mean ± SD (s)")
team_sp = summary.pivot(index="member", columns="executors", values="speedup").round(3)
team_view = team.join(team_sp, lsuffix=" time", rsuffix=" speedup")
print("Wall-clock mean ± SD, then speedup T1/Tn:")
show(team_view)
print("Detail:")
show(summary[["member", "executors", "mean ± SD (s)", "speedup", "efficiency"]].round(3))


## 9 — Runtime and speedup curves


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for member, g in summary.groupby("member"):
    g = g.sort_values("executors")
    ax[0].errorbar(g["executors"], g["mean"], yerr=g["std"], marker="o", capsize=4, label=member)
    ax[1].plot(g["executors"], g["speedup"], marker="o", label=member)
ax[1].plot(EXECUTOR_LIST, EXECUTOR_LIST, "k--", label="ideal linear")
ax[0].set(title="Wall-clock runtime (mean ± SD)", xlabel="Executors (1 core each)", ylabel="Seconds")
ax[1].set(title="Speedup  T1 / Tn", xlabel="Executors (1 core each)", ylabel="Speedup")
for a in ax:
    a.set_xticks(EXECUTOR_LIST)
    a.grid(alpha=0.3)
    a.legend(fontsize=8)
plt.tight_layout()
plot_path = os.path.join(FIG_DIR, "iot_scaling.png")
fig.savefig(plot_path, dpi=150)
plt.show()
plt.close(fig)
print("Wrote", plot_path)


## 10 — Statistical results
Group profile of each `Attack_type`, the features whose class means diverge the most, and the strongest Pearson pairs. The heatmap is the correlation among the 12 most separating features, not the full 80-column matrix (that matrix is saved as CSV).


In [ ]:
stats = last_metrics["Memory: Correlation"]
profile_df = pd.DataFrame(stats["profile"])
for c in profile_df.columns:
    if c != LABEL_COL and pd.api.types.is_numeric_dtype(profile_df[c]):
        profile_df[c] = profile_df[c].astype(float).round(3)
profile_df["share_%"] = (100 * profile_df["n"] / profile_df["n"].sum()).round(2)
profile_path = os.path.join(OUT_DIR, "iot_group_stats.csv")
profile_df.to_csv(profile_path, index=False)
print("Group summary by Attack_type:")
show(profile_df)

mix_df = pd.DataFrame(stats["proto_service"])
mix_df.to_csv(os.path.join(OUT_DIR, "iot_proto_service_counts.csv"), index=False)

sep_df = pd.DataFrame(stats["separators"])
sep_df["class_mean_spread"] = sep_df["class_mean_spread"].round(4)
sep_df.to_csv(os.path.join(OUT_DIR, "iot_feature_separation.csv"), index=False)
print("\nFeatures with the largest spread of class means (spread / global std):")
show(sep_df.head(12))

pairs_df = pd.DataFrame(stats["top_pairs"])
pairs_df["pearson_r"] = pairs_df["pearson_r"].round(4)
pairs_df.to_csv(os.path.join(OUT_DIR, "iot_pearson_top_pairs.csv"), index=False)
print("\nStrongest Pearson pairs:")
show(pairs_df.head(12))

corr_cols = stats["corr_cols"]
corr = np.array(stats["corr"], dtype=float)
pd.DataFrame(corr, index=corr_cols, columns=corr_cols).to_csv(
    os.path.join(OUT_DIR, "iot_pearson_matrix.csv")
)

heat_names = [r["feature"] for r in stats["separators"][:12] if r["feature"] in corr_cols]
idx = [corr_cols.index(n) for n in heat_names]
sub = corr[np.ix_(idx, idx)]
fig, ax = plt.subplots(figsize=(9.5, 8))
im = ax.imshow(sub, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(heat_names)))
ax.set_yticks(range(len(heat_names)))
ax.set_xticklabels(heat_names, rotation=90, fontsize=8)
ax.set_yticklabels(heat_names, fontsize=8)
ax.set_title("Pearson r among the 12 most class-separating features")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
heat_path = os.path.join(FIG_DIR, "iot_correlation_heatmap.png")
fig.savefig(heat_path, dpi=150)
plt.show()
plt.close(fig)
print("Wrote", heat_path)

top_rate = profile_df.loc[profile_df["avg_flow_pkts_per_sec"].idxmax()]
top_syn = profile_df.loc[profile_df["avg_flow_SYN_flag_count"].idxmax()]
top_flags = profile_df.loc[profile_df["avg_flag_events"].idxmax()]
print(
    f"\n{top_rate[LABEL_COL]} has the highest mean packet rate "
    f"({top_rate['avg_flow_pkts_per_sec']})."
)
print(
    f"{top_syn[LABEL_COL]} has the highest mean SYN count "
    f"({top_syn['avg_flow_SYN_flag_count']})."
)
print(
    f"{top_flags[LABEL_COL]} has the highest mean flag-event count "
    f"({top_flags['avg_flag_events']})."
)


## 11 — Classification results
Official metrics come from `MulticlassClassificationEvaluator`. Macro F1 weights every class equally, so it is the number to read next to accuracy on this imbalanced file.


In [ ]:
clf = last_metrics["Sibonginkosi: Logistic Reg."]
print(f"Train rows: {clf['train_rows']:,}    Test rows: {clf['test_rows']:,}    Feature dim: {clf['feature_dim']}")
print(f"Majority-class baseline ({clf['majority_class']}): {clf['baseline_accuracy']:.4f}")
print(f"Accuracy:            {clf['accuracy']:.4f}")
print(f"Weighted precision:  {clf['weighted_precision']:.4f}")
print(f"Weighted recall:     {clf['weighted_recall']:.4f}")
print(f"Weighted F1:         {clf['weighted_f1']:.4f}")
print(f"Macro F1:            {clf['macro_f1']:.4f}")
print(f"Test rows inside the label index: {clf['test_rows_in_matrix']:,} of {clf['test_rows']:,}")

report = pd.DataFrame(clf["per_class"])
for c in ("precision", "recall", "f1"):
    report[c] = report[c].round(4)
report.to_csv(os.path.join(OUT_DIR, "iot_classification_report.csv"), index=False)
print("\nPer-class test metrics:")
show(report)

labels = clf["labels"]
cm = np.array(clf["confusion"], dtype=int)
pd.DataFrame(cm, index=labels, columns=labels).to_csv(os.path.join(OUT_DIR, "iot_confusion_matrix.csv"))
metrics_out = {k: clf[k] for k in [
    "train_rows", "test_rows", "feature_dim", "accuracy", "weighted_f1",
    "weighted_precision", "weighted_recall", "macro_f1", "baseline_accuracy", "majority_class",
]}
with open(os.path.join(OUT_DIR, "iot_ml_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(metrics_out, f, indent=2)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=90, fontsize=8)
ax.set_yticklabels(labels, fontsize=8)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Test confusion matrix")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        if cm[i, j] == 0:
            continue
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=6,
                color="white" if cm[i, j] > cm.max() / 2 else "black")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
cm_path = os.path.join(FIG_DIR, "iot_confusion_matrix.png")
fig.savefig(cm_path, dpi=150)
plt.show()
plt.close(fig)
print("Wrote", cm_path)


## 12 — What the numbers mean for the report
This submitted run has 2 logical CPUs (`os.cpu_count()`) and 13.61 GB RAM. The worker still advertises 4 cores so the session can request 4 executors, but those executors share 2 CPUs. Eight partitions give two executors about four tasks each and four executors about two tasks each. Scheduling and the shuffles in `groupBy`, `Correlation.corr`, and `randomSplit` are a large share of a 123,117-row job. Every timed module re-reads the CSV, repartitions to 8, and re-runs cleaning. Duplicate flows are kept on purpose. No rows were replicated.

No module speeds up. At 2 executors every core is already in use, and wall-clock time still rises: preprocessing speedup 0.78, logistic regression 0.68, correlation 0.62. Correlation drops the most because the Pearson matrix is one large shuffle. Preprocessing drops the least because the CSV read, indexing, and the wide assembler dominate. Logistic regression does more CPU work per row, but the 80/20 split and the feature pipeline still shuffle, and a second core is the most this machine can apply. At 4 executors the job asks for twice as many cores as the machine has, so every module slows further (speedups 0.43, 0.41, and 0.40). Preprocessing’s 1-executor SD is ±11.94 s because the first repeat took 63.3 s and the next two took 41.3 s and 44.3 s.


In [ ]:
print(f"Hardware: {hw['os.cpu_count']} logical CPUs, {hw['ram_gb']} GB RAM, {NUM_PARTITIONS} partitions, no replication.")
pre = last_metrics["Beven: Preprocessing"]
print(
    f"Preprocessing kept {pre['rows_clean']:,} of {pre['rows_raw']:,} rows "
    f"({pre['rows_removed']} removed), scaled {pre['n_numeric']} numeric columns, "
    f"and assembled a {pre['feature_dim']}-dimensional feature vector. "
    f"Label index 0 is {pre['label_order'][0]}."
)
for _, r in summary.sort_values(["member", "executors"]).iterrows():
    print(
        f"{r['member']}: {int(r['executors'])} executors -> {r['mean ± SD (s)']} s, "
        f"speedup {r['speedup']:.3f}, efficiency {r['efficiency']:.3f}"
    )
gain = clf["accuracy"] - clf["baseline_accuracy"]
print(
    f"Logistic regression accuracy is {clf['accuracy']:.3f} against a "
    f"{clf['baseline_accuracy']:.3f} majority baseline ({gain:+.3f}). "
    f"Weighted F1 is {clf['weighted_f1']:.3f} and macro F1 is {clf['macro_f1']:.3f}."
)
weak = report.sort_values("recall").head(3)
print("Lowest test recall:")
for _, r in weak.iterrows():
    print(f"  {r['class']} ({r['role']}): recall {r['recall']:.3f} on {int(r['support'])} test rows")
